In [0]:
# ==============================================================================
# CAMADA BRONZE: Ingestão de Dados Brutos (Inside Airbnb RJ)
# ==============================================================================

from pyspark.sql.functions import current_timestamp, col

# 1. Definição do caminho do arquivo bruto no Volume e nome da tabela de destino
raw_path = "/Volumes/workspace/default/raw_data/listings.csv.gz"
target_table = "workspace.default.bronze_listings"

print("Iniciando a leitura do arquivo bruto em formato CSV...")

# 2. Leitura do arquivo CSV com suporte à compressão gzip e tratamento de multilinhas
df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiline", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(raw_path)
)

# 3. Adição de metadados de governança (data/hora de ingestão e arquivo de origem)
df_bronze = df_raw.withColumn(
    "_ingestion_timestamp", current_timestamp()
).withColumn("_source_file", col("_metadata.file_path"))

# 4. Gravação da tabela Delta na Camada Bronze (Sobrescrita controlada)
print(f"Persistindo os dados na tabela Delta: {target_table}...")
df_bronze.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(target_table)

print(
    "✅ Ingestão concluída com sucesso! Exibindo amostra da tabela Bronze:"
)

# 5. Exibição dos dados salvos e contagem total de registros
display(spark.table(target_table))
print(f"Total de registros ingeridos: {spark.table(target_table).count()}")